# 🏗️ Building Agentic RAG on AWS## Knowledge Bases • Context Graphs • Agentic AI**Duration**: 90 minutes | **Cost**: ~$10-15 USD### The StoryAWS CapEx teams manage **$18B+ in infrastructure transactions** annually. Reconciling across SC.os, AP, OFA, FASL+, and AFS is fragmented and decision context is lost every quarter.We'll build an **AI-powered Metrics Assistant** that:1. **Retrieves** metric definitions with structure-preserving chunking (Module 1)2. **Reasons** over a context graph of decisions and relationships (Module 2)3. **Acts** — classifies risk, records decisions, captures email decisions (Module 3)### Prerequisites- AWS account with admin access- Bedrock model access enabled for **Claude 3.5 Sonnet** and **Titan Embeddings V2**> ⚠️ Run cells in order. Each cell builds on the previous one.

## 🔧 Step 0: Install Dependencies & Configure Credentials

In [ ]:
# Install all dependencies (run once)
%pip install -q boto3 strands-agents strands-agents-tools rich opensearchpy aws-cdk-lib constructs

In [ ]:
# Configure your AWS credentials
# Option A: If running in SageMaker/Cloud9, credentials are automatic
# Option B: Set them explicitly here:

import os
# os.environ["AWS_ACCESS_KEY_ID"] = "YOUR_KEY"
# os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_SECRET"
# os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# Verify credentials work
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
ACCOUNT_ID = identity["Account"]
REGION = boto3.session.Session().region_name or "us-east-1"
print(f"✅ Authenticated as: {identity['Arn']}")
print(f"   Account: {ACCOUNT_ID}")
print(f"   Region: {REGION}")

## 🚀 Step 1: Deploy InfrastructureThis deploys via CloudFormation (generated from CDK):- **S3 bucket** for documents- **OpenSearch Serverless** collection (vector store)- **IAM roles** for Bedrock KB and Lambda- **OpenSearch vector index** (custom resource)Takes ~5 minutes (OpenSearch Serverless collection creation).

In [ ]:
import boto3, json, time

cfn = boto3.client("cloudformation", region_name=REGION)
STACK_NAME = "agentic-rag-workshop"

# Check if stack already exists
try:
    cfn.describe_stacks(StackName=STACK_NAME)
    print(f"Stack {STACK_NAME} already exists. Fetching outputs...")
except cfn.exceptions.ClientError:
    print("Deploying infrastructure stack... (this takes ~5 minutes)")

    # Read the CFN template from the repo, or use inline
    import urllib.request
    # Use the CFN template directly — simpler than CDK in a notebook
    template_url = None

    # Generate minimal CFN inline
    template = json.dumps({
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": "Agentic RAG Workshop Infrastructure",
        "Resources": {
            "DocumentBucket": {
                "Type": "AWS::S3::Bucket",
                "Properties": {
                    "BucketName": f"{STACK_NAME}-docs-{ACCOUNT_ID}",
                    "BucketEncryption": {"ServerSideEncryptionConfiguration": [{"ServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]},
                    "PublicAccessBlockConfiguration": {"BlockPublicAcls": True, "BlockPublicPolicy": True, "IgnorePublicAcls": True, "RestrictPublicBuckets": True}
                }
            },
            "EncPolicy": {
                "Type": "AWS::OpenSearchServerless::SecurityPolicy",
                "Properties": {"Name": f"{STACK_NAME}-enc", "Type": "encryption",
                    "Policy": json.dumps({"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{STACK_NAME}"]}], "AWSOwnedKey": True})}
            },
            "NetPolicy": {
                "Type": "AWS::OpenSearchServerless::SecurityPolicy",
                "Properties": {"Name": f"{STACK_NAME}-net", "Type": "network",
                    "Policy": json.dumps([{"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{STACK_NAME}"]}, {"ResourceType": "dashboard", "Resource": [f"collection/{STACK_NAME}"]}], "AllowFromPublic": True}])}
            },
            "KBRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "RoleName": f"{STACK_NAME}-kb-role",
                    "AssumeRolePolicyDocument": {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"}, "Action": "sts:AssumeRole"}]},
                    "Policies": [{"PolicyName": "kb", "PolicyDocument": {"Version": "2012-10-17", "Statement": [
                        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:ListBucket"], "Resource": [f"arn:aws:s3:::{STACK_NAME}-docs-{ACCOUNT_ID}", f"arn:aws:s3:::{STACK_NAME}-docs-{ACCOUNT_ID}/*"]},
                        {"Effect": "Allow", "Action": ["bedrock:InvokeModel"], "Resource": [f"arn:aws:bedrock:{REGION}::foundation-model/*"]},
                        {"Effect": "Allow", "Action": ["aoss:APIAccessAll"], "Resource": [f"arn:aws:aoss:{REGION}:{ACCOUNT_ID}:collection/*"]},
                        {"Effect": "Allow", "Action": ["lambda:InvokeFunction"], "Resource": [f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:workshop-custom-chunker"]}
                    ]}}]
                }
            },
            "LambdaRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "RoleName": f"{STACK_NAME}-lambda-role",
                    "AssumeRolePolicyDocument": {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]},
                    "ManagedPolicyArns": ["arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"],
                    "Policies": [{"PolicyName": "workshop", "PolicyDocument": {"Version": "2012-10-17", "Statement": [
                        {"Effect": "Allow", "Action": ["s3:*"], "Resource": [f"arn:aws:s3:::{STACK_NAME}-docs-{ACCOUNT_ID}", f"arn:aws:s3:::{STACK_NAME}-docs-{ACCOUNT_ID}/*"]},
                        {"Effect": "Allow", "Action": ["bedrock:*", "bedrock-agent:*"], "Resource": "*"},
                        {"Effect": "Allow", "Action": ["aoss:APIAccessAll"], "Resource": [f"arn:aws:aoss:{REGION}:{ACCOUNT_ID}:collection/*"]},
                        {"Effect": "Allow", "Action": ["neptune-graph:*"], "Resource": "*"}
                    ]}}]
                }
            },
            "DataAccessPolicy": {
                "Type": "AWS::OpenSearchServerless::AccessPolicy",
                "Properties": {"Name": f"{STACK_NAME}-access", "Type": "data",
                    "Policy": json.dumps([{"Rules": [
                        {"ResourceType": "collection", "Resource": [f"collection/{STACK_NAME}"], "Permission": ["aoss:CreateCollectionItems", "aoss:UpdateCollectionItems", "aoss:DescribeCollectionItems"]},
                        {"ResourceType": "index", "Resource": [f"index/{STACK_NAME}/*"], "Permission": ["aoss:CreateIndex", "aoss:UpdateIndex", "aoss:DescribeIndex", "aoss:ReadDocument", "aoss:WriteDocument"]}
                    ], "Principal": [f"arn:aws:iam::{ACCOUNT_ID}:role/{STACK_NAME}-kb-role", f"arn:aws:iam::{ACCOUNT_ID}:role/{STACK_NAME}-lambda-role", identity["Arn"]]}])}
            },
            "Collection": {
                "Type": "AWS::OpenSearchServerless::Collection",
                "DependsOn": ["EncPolicy", "NetPolicy", "DataAccessPolicy"],
                "Properties": {"Name": STACK_NAME, "Type": "VECTORSEARCH", "Description": "Vector store for Bedrock KB"}
            }
        },
        "Outputs": {
            "Bucket": {"Value": {"Ref": "DocumentBucket"}},
            "CollectionArn": {"Value": {"Fn::GetAtt": ["Collection", "Arn"]}},
            "CollectionEndpoint": {"Value": {"Fn::GetAtt": ["Collection", "CollectionEndpoint"]}},
            "KBRoleArn": {"Value": {"Fn::GetAtt": ["KBRole", "Arn"]}},
            "LambdaRoleArn": {"Value": {"Fn::GetAtt": ["LambdaRole", "Arn"]}}
        }
    })

    cfn.create_stack(StackName=STACK_NAME, TemplateBody=template, Capabilities=["CAPABILITY_NAMED_IAM"])

    print("Waiting for stack creation...")
    waiter = cfn.get_waiter("stack_create_complete")
    waiter.wait(StackName=STACK_NAME, WaiterConfig={"Delay": 15, "MaxAttempts": 40})
    print("✅ Stack created!")

# Fetch outputs
outputs = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]}
BUCKET = outputs["Bucket"]
COLLECTION_ARN = outputs["CollectionArn"]
COLLECTION_ENDPOINT = outputs["CollectionEndpoint"]
KB_ROLE_ARN = outputs["KBRoleArn"]
LAMBDA_ROLE_ARN = outputs["LambdaRoleArn"]

print(f"\n📦 Bucket: {BUCKET}")
print(f"🔍 OpenSearch: {COLLECTION_ENDPOINT}")
print(f"🔑 KB Role: {KB_ROLE_ARN}")

### Create the OpenSearch vector indexThe collection is ready but we need to create the k-NN index for vector search.

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth

credentials = boto3.Session().get_credentials()
auth = AWSV4SignerAuth(credentials, REGION, "aoss")

host = COLLECTION_ENDPOINT.replace("https://", "")
client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=auth, use_ssl=True, verify_certs=True,
    connection_class=RequestsHttpConnection, timeout=60,
)

INDEX_NAME = "workshop-kb-index"
try:
    client.indices.get(INDEX_NAME)
    print(f"Index {INDEX_NAME} already exists")
except Exception:
    client.indices.create(INDEX_NAME, body={
        "settings": {"index.knn": True, "number_of_shards": 2, "number_of_replicas": 0},
        "mappings": {"properties": {
            "embedding": {"type": "knn_vector", "dimension": 1024, "method": {"engine": "faiss", "name": "hnsw", "space_type": "l2"}},
            "text": {"type": "text"},
            "metadata": {"type": "text"}
        }}
    })
    print(f"✅ Created vector index: {INDEX_NAME}")

---# 📚 Module 1: Knowledge Base + Custom Chunking (30 min)> **Concept**: Standard RAG uses fixed-size chunking (300 tokens). This splits metric definitions mid-definition — the agent returns partial answers. Custom chunking preserves namespace/metric structure and adds metadata.>> **What we solve**: The `po_aging_not_invoiced` metric has type, filters, dimensions, and periods. Fixed chunking splits this across two chunks. Custom chunking keeps it intact.

### Step 1: Create sample AFS documents and upload to S3

In [ ]:
# AFS sample documents — inline so the notebook is self-contained
DOCUMENTS = {
"metric-definitions.md": '''# AFS Integrity Metrics — Metric Definitions

## Namespace: veritas_po_metrics

### Metric: po_total_spend
- **Type**: SUM
- **Value Column**: po_amount
- **Breakdown Column**: currency
- **Period Column**: po_received_at
- **Periods**: MTD, QTD, YTD
- **Dimensions**: entity, region, cost_center
- **Filters**: po_amount > 300, line_category IN [AHU, GENERATOR]
- **Collections**: po_number

### Metric: po_aging_not_invoiced
- **Type**: COUNT, SUM
- **Value Column**: po_amount
- **Periods**: MTD, QTD
- **Dimensions**: entity, currency, cost_center
- **Filters**: days_since_receipt > 90, invoice_status = PENDING
- **Description**: POs aging more than 90 days but not yet invoiced.

### Metric: invoice_variance
- **Type**: COUNT, SUM
- **Value Column**: variance_amount
- **Periods**: MTD, QTD, YTD
- **Filters**: variance_pct > 10
- **Description**: Invoices with more than 10% variance vs estimated PO price.

## Namespace: afs_asset_metrics

### Metric: asset_create_count
- **Type**: COUNT
- **Value Column**: asset_id
- **Periods**: MTD, QTD, YTD
- **Dimensions**: entity, asset_category, region

### Metric: failed_asset_creates
- **Type**: COUNT, SUM
- **Value Column**: asset_amount
- **Periods**: MTD, QTD
- **Dimensions**: entity, asset_category, failure_reason
- **Filters**: status = FAILED

### Metric: asset_value_mismatch
- **Type**: COUNT, SUM
- **Filters**: ofa_value != afs_value
- **Description**: Assets where value does not tie out between AFS and OFA.

## Namespace: lle_reconciliation

### Metric: afs_vs_ofa_coverage
- **Type**: SUM
- **Breakdown Column**: system_source
- **Periods**: MTD, QTD, YTD
- **Description**: Proportion of costs processed by AFS vs total OFA balance for LLE.
''',

"reconciliation-rules.md": '''# AFS Reconciliation Rules

## PO to Receipt Matching
- **Match Keys**: po_number, line_number
- **Tolerance**: Receipt quantity within 5% of PO quantity
- **Aging**: POs without receipts after 90 days flagged (was 60 days, changed per Q4-2024 audit — 40% false positives at 60 days)

## Receipt to Invoice Matching
- **Match Keys**: po_number, receipt_number
- **Tolerance**: Invoice amount within 10% of PO estimated price
- **Three-way match required**: PO, Receipt, Invoice

## AFS to OFA Asset Reconciliation
- All AFS-created assets must appear in OFA within 2 business days
- Failed creates tracked via failed_asset_creates metric
- Asset value in AFS must match invoice amount after FX conversion (0.5% tolerance)

## Period Close Procedures
1. Run all namespace metric refreshes
2. Review HIGH risk items (>$500K or aging >120 days)
3. Clear or escalate mismatches >$10,000
4. Generate period snapshot for audit trail

## Escalation Matrix
- **SEV-1**: Metric generation failure during close → 15 min response
- **SEV-2**: Reconciliation variance >$1M → 30 min response
- **SEV-3**: Variance >$10K → 4 hours
- **SEV-4**: Minor data quality → next business day
''',

"namespace-configs.md": '''# AFS Namespace Configurations

## Decision Log

### DEC-003: PO aging threshold (60 → 90 days)
- **Date**: 2024-10-15
- **Decided by**: Sarah Chen, FBI CapEx Lead
- **Decision**: Changed po_aging_not_invoiced threshold from 60 to 90 days
- **Rationale**: Q4 2024 audit found 60-day threshold generated ~40% false positives
- **Impact**: Reduced false positives by ~35%

### DEC-004: LLE category reclassification
- **Date**: 2024-07-01
- **Decided by**: Michael Torres, Finance Director
- **Decision**: Added GENERATOR to LLE line categories alongside AHU
- **Rationale**: FY2024 GAAP update required reclassification
- **Impact**: Increased LLE transaction volume by ~15%

### DEC-005: Capitalization aging (120 → 180 days)
- **Date**: 2024-08-01
- **Decision**: Extended capitalization aging threshold from 120 to 180 days
- **Rationale**: Complex infrastructure assets have longer commissioning timelines
- **Impact**: Reduced false aging alerts by ~25%

### DEC-001: DataStudio Redshift over LakeFormation
- **Date**: 2025-01-15
- **Decided by**: Raj Patel, AFS Engineering Lead
- **Decision**: Use DataStudio Redshift for metric query compute
- **Rationale**: Customers can test queries in DataStudio before onboarding to metrics framework
''',

"compliance-policies.md": '''# AFS Compliance & Audit Policies

## GRC Controls
| Control | Description | Frequency | Owner |
|---------|-------------|-----------|-------|
| GRC-001 | OFA to FASL+ recon variance < $10K | Monthly | FBI CapEx |
| GRC-002 | All POs invoiced within 90 days or escalated | Monthly | Accounting |
| GRC-003 | Asset create success rate > 99.5% | Monthly | AFS Engineering |
| GRC-004 | ERV variance < 5% for 95% of assets | Quarterly | Finance |

## Data Retention
- Metric outputs: 7 years (SOX)
- Transaction details: 7 years
- Decision traces: 7 years
- Intermediate data: 90 days then S3 Glacier

## Close Calendar
| Period | Close Deadline | Metric Refresh | Certification |
|--------|---------------|----------------|---------------|
| Month-End | BD+3 | BD+1 | BD+2 |
| Quarter-End | BD+5 | BD+2 | BD+4 |
| Year-End | BD+10 | BD+3 | BD+8 |
''',

"decision-emails.md": '''# AFS Decision Emails

## Email: PO Aging Threshold Review
**From**: Sarah Chen (FBI CapEx Lead) | **Date**: 2024-10-15
After reviewing Q4 audit findings, I approve changing PO aging threshold from 60 to 90 days.
The 60-day threshold generated ~40% false positives during Q3 close. 90 days better reflects procurement timelines. Expected ~35% reduction in false alerts.

## Email: GENERATOR Category
**From**: Michael Torres (Finance Director) | **Date**: 2024-07-01
Per FY2024 GAAP update, generator assets must be classified under LLE alongside AHU.
This increases LLE transaction volume by ~15%.

## Email: March Failed Asset Creates
**From**: Lisa Wang (FBI CapEx Analyst) | **Date**: 2025-03-18
Investigated spike in failed creates (23 vs 8 last month). Root cause: 15 of 23 are GENERATOR assets onboarded March 15. OFA interface not updated for new sub-category codes. Hotfix deployed March 18.
'''
}

# Upload to S3
s3 = boto3.client("s3", region_name=REGION)
for name, content in DOCUMENTS.items():
    s3.put_object(Bucket=BUCKET, Key=f"documents/{name}", Body=content.encode())
    print(f"  Uploaded {name}")
print(f"\n✅ {len(DOCUMENTS)} documents uploaded to s3://{BUCKET}/documents/")

### Step 2: Create the Bedrock Knowledge Base

In [ ]:
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)

EMBED_MODEL = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2:0"

print("Creating Knowledge Base...")
kb = bedrock_agent.create_knowledge_base(
    name="afs-metrics-kb",
    description="AFS Integrity Metrics — definitions, recon rules, policies",
    roleArn=KB_ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {"embeddingModelArn": EMBED_MODEL},
    },
    storageConfiguration={
        "type": "OPENSEARCH_SERVERLESS",
        "opensearchServerlessConfiguration": {
            "collectionArn": COLLECTION_ARN,
            "vectorIndexName": INDEX_NAME,
            "fieldMapping": {"vectorField": "embedding", "textField": "text", "metadataField": "metadata"},
        },
    },
)
KB_ID = kb["knowledgeBase"]["knowledgeBaseId"]
print(f"KB created: {KB_ID}")

# Create data source with fixed-size chunking (baseline)
ds = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID, name="afs-documents",
    dataSourceConfiguration={"type": "S3", "s3Configuration": {"bucketArn": f"arn:aws:s3:::{BUCKET}"}},
    vectorIngestionConfiguration={"chunkingConfiguration": {
        "chunkingStrategy": "FIXED_SIZE",
        "fixedSizeChunkingConfiguration": {"maxTokens": 300, "overlapPercentage": 20},
    }},
)
DS_ID = ds["dataSource"]["dataSourceId"]

# Start ingestion
job = bedrock_agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
job_id = job["ingestionJob"]["ingestionJobId"]
print("Ingesting documents...")
while True:
    status = bedrock_agent.get_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID, ingestionJobId=job_id)
    state = status["ingestionJob"]["status"]
    print(f"  {state}")
    if state in ("COMPLETE", "FAILED"): break
    time.sleep(5)

stats = status["ingestionJob"].get("statistics", {})
print(f"\n✅ Ingestion complete! Docs indexed: {stats.get('numberOfNewDocumentsIndexed', 'N/A')}")

# Save for later modules
os.environ["KNOWLEDGE_BASE_ID"] = KB_ID

### Step 3: Query the KB — see what works and what doesn't

In [ ]:
bedrock_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)
MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-3-5-sonnet-20241022-v2:0"

def query_kb(question, verbose=False):
    """Query the KB and optionally show chunk details."""
    print(f"🔍 {question}\n")
    response = bedrock_runtime.retrieve_and_generate(
        input={"text": question},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {"knowledgeBaseId": KB_ID, "modelArn": MODEL_ARN},
        },
    )
    print(f"💬 {response['output']['text']}\n")
    if verbose:
        chunks = bedrock_runtime.retrieve(
            knowledgeBaseId=KB_ID, retrievalQuery={"text": question},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
        )
        for i, r in enumerate(chunks["retrievalResults"], 1):
            print(f"  Chunk {i} [score: {r.get('score',0):.3f}]: {r['content']['text'][:120]}...")

# Try queries
query_kb("What metrics track PO aging for the LLE category?", verbose=True)
print("=" * 60)
query_kb("Why was the PO aging threshold changed from 60 to 90 days?")

### Step 4: Compare chunking strategies (local demo)This runs locally — no AWS calls — to show how different strategies split the same document.

In [ ]:
import re

doc = DOCUMENTS["metric-definitions.md"]

# Fixed-size: splits blindly
fixed = []
words = doc.split()
chunk, count = [], 0
for w in words:
    chunk.append(w); count += 1
    if count >= 60:  # ~300 tokens
        fixed.append(" ".join(chunk)); chunk = chunk[-12:]; count = 12
if chunk: fixed.append(" ".join(chunk))

# Namespace-aware: splits on ## headers, tracks namespace
namespace_chunks = []
current_ns = "unknown"
for section in re.split(r"\n(?=##+ )", doc):
    section = section.strip()
    if not section or len(section) < 20: continue
    ns = re.search(r"Namespace:\s*(\w+)", section)
    if ns: current_ns = ns.group(1)
    metric = re.search(r"Metric:\s*(\w+)", section)
    namespace_chunks.append({
        "text": section,
        "namespace": current_ns,
        "metric": metric.group(1) if metric else "",
    })

print(f"Fixed-size chunking: {len(fixed)} chunks")
print(f"  Chunk 1: {fixed[0][:100]}...")
print(f"  Chunk 2: {fixed[1][:100]}...")
print(f"\nNamespace-aware chunking: {len(namespace_chunks)} chunks")
for c in namespace_chunks[:4]:
    label = f"[{c['namespace']}] {c['metric']}" if c['metric'] else c['namespace']
    print(f"  {label}: {c['text'][:80]}...")

print("\n⚠️  Notice: fixed-size may split a metric definition across chunks!")
print("✅  Namespace-aware keeps each metric definition intact with metadata.")

---# 🔗 Module 2: Context Graph (30 min)> **Concept**: A context graph stores entities and relationships in a graph database with vector search. This enables **GraphRAG** — hybrid retrieval combining vector similarity with graph traversal. It's the "event clock" that captures WHY decisions were made.>> **What we solve**: "Why is the PO aging threshold 90 days?" — the KB quotes the config, but the graph returns Decision DEC-003 with rationale, approver, and linked metrics.

### Step 1: Create Neptune Analytics graph

In [ ]:
neptune = boto3.client("neptune-graph", region_name=REGION)

GRAPH_NAME = "workshop-context-graph"

# Check if graph exists
existing = neptune.list_graphs().get("graphs", [])
graph_match = [g for g in existing if g.get("name") == GRAPH_NAME]

if graph_match:
    GRAPH_ID = graph_match[0]["id"]
    print(f"Graph already exists: {GRAPH_ID}")
else:
    print("Creating Neptune Analytics graph (takes 5-10 min)...")
    resp = neptune.create_graph(
        graphName=GRAPH_NAME, provisionedMemory=32,
        vectorSearchConfiguration={"dimension": 1024},
        publicConnectivity=False, replicaCount=0, deletionProtection=False,
    )
    GRAPH_ID = resp["id"]
    print(f"Graph ID: {GRAPH_ID}")

    while True:
        status = neptune.get_graph(graphIdentifier=GRAPH_ID)
        state = status["status"]
        print(f"  {state}")
        if state == "AVAILABLE": break
        if state in ("FAILED", "DELETING"):
            raise Exception(f"Graph creation failed: {state}")
        time.sleep(30)

print(f"\n✅ Graph ready: {GRAPH_ID}")
GRAPH_ENDPOINT = neptune.get_graph(graphIdentifier=GRAPH_ID)["endpoint"]

### Step 2: Populate the graph with entities and decisionsWe use Claude to extract entities and relationships from the AFS documents, then write them to Neptune.

In [ ]:
bedrock_rt = boto3.client("bedrock-runtime", region_name=REGION)

def get_embedding(text):
    resp = bedrock_rt.invoke_model(modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text[:8000]}))
    return json.loads(resp["body"].read())["embedding"]

def run_query(cypher):
    try:
        resp = neptune.execute_query(graphIdentifier=GRAPH_ID, queryString=cypher, language="OPEN_CYPHER")
        return json.loads(resp["payload"].read())
    except Exception as e:
        return {"error": str(e), "results": []}

# Extract entities from each document using Claude
for doc_name, doc_text in DOCUMENTS.items():
    print(f"\n📄 Processing: {doc_name}")

    prompt = f"""Extract entities and relationships from this AFS financial document.
Document: {doc_name}
---
{doc_text[:5000]}
---
Return ONLY valid JSON:
{{"entities": [{{"name": "str", "type": "System|Metric|Decision|Policy|Dimension", "description": "str"}}],
 "relationships": [{{"source": "str", "target": "str", "type": "DEFINES|TRACKS|DECIDED|AFFECTS|FEEDS_INTO", "description": "str"}}],
 "decisions": [{{"id": "str", "title": "str", "date": "str", "rationale": "str"}}]}}"""

    resp = bedrock_rt.invoke_model(modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
        body=json.dumps({"anthropic_version": "bedrock-2023-05-31", "max_tokens": 4096,
            "messages": [{"role": "user", "content": prompt}]}))
    content = json.loads(resp["body"].read())["content"][0]["text"]
    start, end = content.find("{"), content.rfind("}") + 1
    extracted = json.loads(content[start:end])

    safe = lambda s: str(s).replace("'", "").replace("\\", "")[:200]

    for e in extracted.get("entities", [])[:15]:
        emb = get_embedding(f"{e['name']}: {e.get('description','')}")
        run_query(f"MERGE (n:{e['type']} {{name: '{safe(e['name'])}'}}) ON CREATE SET n.description = '{safe(e.get('description',''))}', n.embedding = {emb}")
        print(f"  + [{e['type']}] {e['name']}")

    for r in extracted.get("relationships", [])[:15]:
        run_query(f"MATCH (s {{name: '{safe(r['source'])}'}}) MATCH (t {{name: '{safe(r['target'])}'}}) MERGE (s)-[:{r['type']}]->(t)")
        print(f"  → {r['source']} --[{r['type']}]--> {r['target']}")

    for d in extracted.get("decisions", [])[:5]:
        emb = get_embedding(f"{d['title']}: {d.get('rationale','')}")
        run_query(f"MERGE (d:Decision {{id: '{safe(d['id'])}'}}) ON CREATE SET d.title = '{safe(d['title'])}', d.date = '{safe(d.get('date',''))}', d.rationale = '{safe(d.get('rationale',''))}', d.embedding = {emb}")
        print(f"  ★ Decision: {d['title']}")

print("\n✅ Graph populated!")

### Step 3: Query the graph — vector + graph hybrid

In [ ]:
question = "Why was the PO aging threshold changed from 60 to 90 days?"
print(f"🔍 {question}\n")

q_emb = get_embedding(question)

# Vector search in graph
print("🔗 Graph Vector Search:")
results = run_query(f"""
    CALL neptune.algo.vectors.topKByNode({{queryVector: {q_emb}, topK: 5}})
    YIELD node, score
    OPTIONAL MATCH (node)-[r]-(connected)
    RETURN labels(node) AS type, node.name AS name, node.description AS desc, score,
           collect(DISTINCT {{rel: type(r), target: connected.name}})[..3] AS connections
    ORDER BY score DESC
""")
for r in results.get("results", []):
    print(f"  [{r.get('score',0):.3f}] ({','.join(r.get('type',[]))}) {r.get('name','')}: {str(r.get('desc',''))[:80]}")
    for c in r.get("connections", []):
        if c.get("target"): print(f"    → {c['rel']}: {c['target']}")

# Decision search
print("\n⚖️ Related Decisions:")
decisions = run_query(f"""
    CALL neptune.algo.vectors.topKByNode({{queryVector: {q_emb}, topK: 3, nodeLabels: ['Decision']}})
    YIELD node, score
    RETURN node.id AS id, node.title AS title, node.rationale AS rationale, score
    ORDER BY score DESC
""")
for d in decisions.get("results", []):
    print(f"  [{d.get('score',0):.3f}] {d.get('id','')}: {d.get('title','')}")
    print(f"    Rationale: {str(d.get('rationale',''))[:120]}")

print("\n✅ The graph returns WHY (decisions) alongside WHAT (current config)")

---# 🤖 Module 3: Agentic AI with Strands (30 min)> **Concept**: An agent is an LLM with tools. **Strands Agents SDK** uses a model-driven approach — you give the LLM tools (Python functions with `@tool`) and it decides when to call them. The agent loops until it has enough information to answer.>> **What we solve**: The assistant can now search KB, query graph, classify risk, record decisions, AND capture decisions from email threads — all in natural conversation.

### Step 1: Simple agent with KB retrieval

In [ ]:
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands_tools import retrieve

os.environ["KNOWLEDGE_BASE_ID"] = KB_ID

model = BedrockModel(model_id="anthropic.claude-3-5-sonnet-20241022-v2:0", region_name=REGION)

simple_agent = Agent(
    model=model, tools=[retrieve],
    system_prompt="You are the AFS Metrics Assistant. Use retrieve to search the knowledge base for metric definitions, reconciliation rules, and policies. Cite specific metrics and thresholds.",
)

print("🤖 Simple Agent — KB retrieval only\n")
result = simple_agent("What metrics track PO aging for the LLE category?")
print(f"\n💬 {result}")

### Step 2: Add custom AFS tools

In [ ]:
@tool
def classify_risk(po_count: int, total_amount: float, aging_days: int) -> str:
    """Classify financial risk level for POs or assets.

    Args:
        po_count: Number of items.
        total_amount: Total dollar amount.
        aging_days: Days aging.
    """
    if total_amount > 500000 or aging_days > 120:
        return f"HIGH RISK: {po_count} items, ${total_amount:,.0f}, {aging_days}d. Escalate to FBI CapEx lead within 24h."
    if total_amount > 100000 or aging_days > 90:
        return f"MEDIUM RISK: {po_count} items, ${total_amount:,.0f}, {aging_days}d. Review within 48h."
    return f"LOW RISK: {po_count} items, ${total_amount:,.0f}, {aging_days}d. Standard processing."

@tool
def get_period_info() -> str:
    """Get current accounting period and close deadlines."""
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    return f"Period: {now.strftime('%Y-%m')} (Q{(now.month-1)//3+1})\nMonth-end close: BD+3\nMetric refresh: BD+1"

@tool
def check_grc_control(control_id: str) -> str:
    """Check GRC control details.

    Args:
        control_id: Control ID like GRC-001.
    """
    controls = {
        "GRC-001": "OFA to FASL+ recon variance < $10K | Monthly | FBI CapEx",
        "GRC-002": "All POs invoiced within 90 days or escalated | Monthly | Accounting",
        "GRC-003": "Asset create success rate > 99.5% | Monthly | AFS Engineering",
        "GRC-004": "ERV variance < 5% for 95% of assets | Quarterly | Finance",
    }
    return controls.get(control_id, f"Unknown control: {control_id}")

tools_agent = Agent(
    model=model,
    tools=[retrieve, classify_risk, get_period_info, check_grc_control],
    system_prompt="You are the AFS Metrics Assistant. Search KB for policies, classify risk when asked, check period info and GRC controls.",
)

print("🤖 Agent with custom tools\n")
result = tools_agent("We have 47 POs totaling $3.2M aging for 95 days. What's the risk and what GRC control applies?")
print(f"\n💬 {result}")

### Step 3: Add graph tools — the full integrated agent

In [ ]:
# Graph tools — search context graph and record decisions
@tool
def search_context_graph(query: str) -> str:
    """Search the context graph for entities, relationships, and decision history.
    Use this to find WHY things are the way they are.

    Args:
        query: Search query about entities, decisions, or relationships.
    """
    emb = get_embedding(query)
    entities = run_query(f"""
        CALL neptune.algo.vectors.topKByNode({{queryVector: {emb}, topK: 5}})
        YIELD node, score WHERE score > 0.3
        OPTIONAL MATCH (node)-[r]-(c)
        RETURN labels(node) AS type, node.name AS name, node.description AS desc, score,
               collect(DISTINCT {{rel: type(r), target: c.name}})[..3] AS conns
        ORDER BY score DESC""")
    decisions = run_query(f"""
        CALL neptune.algo.vectors.topKByNode({{queryVector: {emb}, topK: 3, nodeLabels: ['Decision']}})
        YIELD node, score WHERE score > 0.3
        RETURN node.id AS id, node.title AS title, node.rationale AS rationale, score
        ORDER BY score DESC""")

    parts = ["## Entities"]
    for r in entities.get("results", []):
        parts.append(f"- [{','.join(r.get('type',[]))}] {r.get('name','')}: {str(r.get('desc',''))[:100]}")
        for c in r.get("conns", []):
            if c.get("target"): parts.append(f"  → {c['rel']}: {c['target']}")
    parts.append("\n## Decisions")
    for r in decisions.get("results", []):
        parts.append(f"- {r.get('id','')}: {r.get('title','')} — {str(r.get('rationale',''))[:120]}")
    return "\n".join(parts) or "No relevant context found."

@tool
def record_decision(query: str, reasoning: str, decision: str, entities_considered: str) -> str:
    """Record an agent decision trace in the context graph.

    Args:
        query: Original user question.
        reasoning: Agent's reasoning process.
        decision: Final recommendation.
        entities_considered: Comma-separated entity names.
    """
    from datetime import datetime, timezone
    trace_id = f"TRACE-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
    emb = get_embedding(f"{query} {reasoning} {decision}")
    safe = lambda s: str(s).replace("'", "")[:200]
    run_query(f"CREATE (t:DecisionTrace {{id: '{trace_id}', user_query: '{safe(query)}', reasoning: '{safe(reasoning)}', decision: '{safe(decision)}', embedding: {emb}}})")
    for e in entities_considered.split(","):
        e = e.strip()
        if e: run_query(f"MATCH (t:DecisionTrace {{id: '{trace_id}'}}) MATCH (n {{name: '{safe(e)}'}}) MERGE (t)-[:CONSIDERED]->(n)")
    return f"Decision trace {trace_id} recorded."

@tool
def ingest_email_decision(email_text: str) -> str:
    """Extract a decision from an email and store it in the context graph.

    Args:
        email_text: Full email text containing a decision.
    """
    from datetime import datetime, timezone
    prompt = ("Extract the decision from this email. Return ONLY valid JSON:\n"
        '{"id": "short-id", "title": "brief", "date": "YYYY-MM-DD", "decided_by": "name", '
        '"decision": "what", "rationale": "why", "entities_affected": ["list"]}\n\n' + email_text[:3000])
    resp = bedrock_rt.invoke_model(modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
        body=json.dumps({"anthropic_version": "bedrock-2023-05-31", "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}]}))
    content = json.loads(resp["body"].read())["content"][0]["text"]
    d = json.loads(content[content.find("{"):content.rfind("}")+1])
    safe = lambda s: str(s).replace("'", "")[:200]
    emb = get_embedding(f"{d.get('title','')} {d.get('rationale','')}")
    run_query(f"MERGE (dec:Decision {{id: '{safe(d.get('id',''))}'}}) ON CREATE SET dec.title = '{safe(d.get('title',''))}', dec.date = '{safe(d.get('date',''))}', dec.decided_by = '{safe(d.get('decided_by',''))}', dec.rationale = '{safe(d.get('rationale',''))}', dec.source = 'email', dec.embedding = {emb}")
    for e in d.get("entities_affected", []):
        run_query(f"MERGE (n:Entity {{name: '{safe(e)}'}})"); run_query(f"MATCH (dec:Decision {{id: '{safe(d.get('id',''))}'}}) MATCH (n {{name: '{safe(e)}'}}) MERGE (dec)-[:AFFECTS]->(n)")
    return f"Captured: {d.get('id','')} — {d.get('title','')} (from email by {d.get('decided_by','')})"

print("✅ All tools defined: retrieve, classify_risk, get_period_info, check_grc_control,")
print("   search_context_graph, record_decision, ingest_email_decision")

### Step 4: The complete AFS Metrics Agent

In [ ]:
full_agent = Agent(
    model=model,
    tools=[retrieve, search_context_graph, record_decision, ingest_email_decision,
           classify_risk, get_period_info, check_grc_control],
    system_prompt=(
        "You are the AFS Metrics Assistant for AWS CapEx financial operations.\n"
        "Tools:\n"
        "- retrieve: Search KB for metric definitions, recon rules, policies (WHAT is true)\n"
        "- search_context_graph: Query graph for decisions and relationships (WHY it's true)\n"
        "- record_decision: Save your reasoning as audit trail\n"
        "- ingest_email_decision: Extract decisions from email threads\n"
        "- classify_risk: Classify financial risk levels\n"
        "- get_period_info: Current accounting period\n"
        "- check_grc_control: GRC control details\n\n"
        "For every question: search KB for facts, query graph for decisions, explain WHAT + WHY."
    ),
)
print("🤖 Full AFS Metrics Agent ready!\n")

# Demo 1: KB + Graph
print("=" * 60)
print("Demo 1: What metrics track PO aging and WHY is the threshold 90 days?")
print("=" * 60)
result = full_agent("What metrics track PO aging for LLE and why is the threshold set to 90 days instead of 60?")
print(f"\n💬 {result}")

### Step 5: Capture a decision from email

In [ ]:
print("=" * 60)
print("Demo 2: Capture decision from email thread")
print("=" * 60)

email = """From: Sarah Chen (FBI CapEx Lead)
Date: 2024-10-15
Subject: RE: PO Aging Threshold Review

After reviewing the Q4 audit findings, I approve changing the PO aging threshold from 60 to 90 days.
The 60-day threshold generated approximately 40% false positives during Q3 close.
90 days better reflects actual procurement timelines. Expected ~35% reduction in false alerts."""

result = full_agent(f"Please capture the decision from this email:\n\n{email}")
print(f"\n💬 {result}")

In [ ]:
# Now query the graph for the captured decision
print("=" * 60)
print("Demo 3: Query graph for PO aging decisions (should find the email!)")
print("=" * 60)
result = full_agent("What decisions have been made about PO aging thresholds? Who approved them?")
print(f"\n💬 {result}")

### Step 6: Month-end close scenario

In [ ]:
print("=" * 60)
print("Demo 4: Month-end close — classify risk and record decision")
print("=" * 60)
result = full_agent(
    "It's month-end close. We have 47 EMEA POs aging >90 days totaling $3.2M. "
    "Classify the risk, tell me which GRC control applies, and record your decision."
)
print(f"\n💬 {result}")

### Step 7: Interactive chat (optional)Uncomment and run to chat interactively with the agent.

In [ ]:
# Uncomment to start interactive chat:
# print("🤖 AFS Metrics Assistant — type 'quit' to exit\n")
# while True:
#     q = input("You: ").strip()
#     if not q or q.lower() in ("quit", "exit"): break
#     result = full_agent(q)
#     print(f"\nAssistant: {result}\n")

---# 🧹 CleanupRun this cell to delete ALL workshop resources and stop incurring charges.

In [ ]:
# ⚠️ This deletes everything — only run when you're done!

print("🧹 Cleaning up workshop resources...\n")

# Delete Bedrock Knowledge Base
try:
    bedrock_agent.delete_knowledge_base(knowledgeBaseId=KB_ID)
    print(f"  ✅ Deleted KB: {KB_ID}")
except Exception as e:
    print(f"  ⚠️ KB: {e}")

# Delete Neptune graph
try:
    neptune.delete_graph(graphIdentifier=GRAPH_ID, skipSnapshot=True)
    print(f"  ✅ Deleted Neptune graph: {GRAPH_ID}")
except Exception as e:
    print(f"  ⚠️ Neptune: {e}")

# Empty S3 bucket
try:
    objects = s3.list_objects_v2(Bucket=BUCKET).get("Contents", [])
    if objects:
        s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in objects]})
    print(f"  ✅ Emptied S3 bucket: {BUCKET}")
except Exception as e:
    print(f"  ⚠️ S3: {e}")

# Delete CloudFormation stack
try:
    cfn.delete_stack(StackName=STACK_NAME)
    print(f"  ✅ Deleting stack: {STACK_NAME} (takes ~3 min)")
    print("     Waiting for deletion...")
    waiter = cfn.get_waiter("stack_delete_complete")
    waiter.wait(StackName=STACK_NAME, WaiterConfig={"Delay": 10, "MaxAttempts": 30})
    print(f"  ✅ Stack deleted!")
except Exception as e:
    print(f"  ⚠️ Stack: {e}")

print("\n✅ All workshop resources cleaned up!")

---## 🎉 Workshop Complete!### What you built:1. **Knowledge Base** with namespace-aware custom chunking — complete metric definitions, not fragments2. **Context Graph** in Neptune Analytics — decisions, relationships, and the event clock3. **Strands Agent** with 7 tools — retrieves, reasons, acts, and records decisions4. **Email Decision Capture** — extracts decisions from email threads into the graph### Key concepts:- **Two Clock Problem**: State clock (WHAT) + Event clock (WHY) — context graphs fill the gap- **GraphRAG**: Vector search + graph traversal fused via Reciprocal Rank Fusion- **Agentic AI**: LLM + tools — model-driven orchestration with `@tool` decorator- **Decision Traces**: Agent reasoning as searchable graph nodes — institutional memory### What's next:- Add Bedrock Guardrails for financial data compliance- Connect to DataStudio Redshift for live metric queries- Deploy as API with Lambda + API Gateway- Multi-agent: separate specialists for PO, AP, Asset, Reconciliation